In [1]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [2]:
%config InlineBackend.figure_format = 'retina'

import random
import time
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from nsppk import NSPPK

from abstractgraph_graphicalizer.chem import draw_molecules as display_graphs

from abstractgraph.operators import *
from abstractgraph.hashing import hash_graph
from abstractgraph.graphs import graph_to_abstract_graph
from abstractgraph.vectorize import AbstractGraphTransformer
from abstractgraph_ml.estimators import GraphEstimator
from abstractgraph_generative.conditional import ConditionalAutoregressiveGenerator
from abstractgraph_generative.conditional_attributed import AttributedConditionalAutoregressiveGenerator
from abstractgraph_generative.conditional_batch import  ConditionalAutoregressiveGraphsGenerator
from abstractgraph_generative.edge_generator import (
    EdgeGenerator,
    edge_neighbors,
    make_edge_regression_dataset,
    remove_edges,
)


/Users/fabriziocosta/miniconda3/envs/py311/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.2.0)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [3]:
from abstractgraph_ml.feasibility import (
    FeasibilityEstimator,
    FeasibilityEstimatorFeatureCannotExist,
)

nbits = 19
feasibility_kwargs = dict(
    nbits=nbits,
    parallel=True,
    backend='loky',
    n_jobs=-1,
)

feasibility_estimators = [
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=compose(neighborhood(radius=2), unlabel()),
        **feasibility_kwargs,
    ),
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=path(number_of_edges=2),
        **feasibility_kwargs,
    ),
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=neighborhood(radius=1),
        **feasibility_kwargs,
    ),
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=cycle(),
        **feasibility_kwargs,
    ),
    FeasibilityEstimatorFeatureCannotExist(
        decomposition_function=compose(combination(number_of_elements=(2, 3), distance=0), cycle(), unlabel()),
        **feasibility_kwargs,
    ),
]

feasibility_estimator = FeasibilityEstimator(feasibility_estimators)

In [ ]:
from abstractgraph_graphicalizer.chem import ZINCLoader

loader = ZINCLoader(on_error="skip")

dataset_name = "zinc_250k"
size = 2000
min_num_nodes = 10
max_num_nodes = 13

graphs, metadata = loader.load(
    dataset_name,
    limit=size,
    min_node_count=min_num_nodes,
    max_node_count=max_num_nodes,
)
print(f"dataset: {dataset_name}")
print(f"size: {len(graphs)}")
print(f"node_range: [{min_num_nodes}, {max_num_nodes}]")

from abstractgraph.utils import plot_graph_label_counts
_ = plot_graph_label_counts(graphs, top=20, title='Dataset info', log_scale=True)

# Keep downstream notebook cells unchanged
all_graphs = graphs
all_targets = metadata
print('dataset size:', len(all_graphs))


dataset: zinc_250k
size: 2000
node_range: [10, 13]
dataset size: 2000


In [ ]:
_ = display_graphs(graphs[:7*2], n_graphs_per_line=7)

In [ ]:
graph= graphs[0]
perturbed_graphs = edge_neighbors(graph, n_samples=6)
_ = display_graphs([graph]+perturbed_graphs, n_graphs_per_line=7)

In [ ]:

n_negative_per_positive = 3
positives, negatives, dataset = make_edge_regression_dataset(
    graph,
    n_negative_per_positive=n_negative_per_positive,
    seed=0,
)

print('n_positives =', len(positives))
print('n_negatives =', len(negatives))
print('dataset_size =', len(dataset))
print('n_terminal_positives =', sum(G.number_of_edges() == 0 for G in positives))

In [ ]:
_ = display_graphs(positives[:7*2], n_graphs_per_line=7)

In [ ]:
_ = display_graphs(negatives[:7*2], n_graphs_per_line=7)

In [ ]:
fit_graphs = random.sample(graphs, k=min(1*4, len(graphs)))
fit_targets = [max(dict(graph.degree()).values(), default=0) for graph in fit_graphs]
print(f'fitting on {len(fit_graphs)} graphs')
print('target values (max_degree):', sorted(set(fit_targets)))
_ = display_graphs(fit_graphs, n_graphs_per_line=7)

In [ ]:
%%time
USE_NSPPK = True

if USE_NSPPK:
    vectorizer = NSPPK(radius=1, distance=4, connector=1, nbits=14, parallel=True)
else:
    df = add(neighborhood(radius=(1,2)), path(number_of_edges=2))
    vectorizer = AbstractGraphTransformer(
        nbits=14,
        decomposition_function=df,
        return_dense=True,
        n_jobs=-1,
    )

graph_estimator = GraphEstimator(
    transformer=vectorizer,
    estimator=RandomForestClassifier(
        random_state=0, 
        n_estimators=300, 
        n_jobs=-1, 
        class_weight="balanced_subsample",
        ),
)

target_estimator = GraphEstimator(
    transformer=vectorizer,
    estimator=RandomForestRegressor(
        random_state=0,
        n_estimators=300,
        n_jobs=-1,
    ),
)

generator = EdgeGenerator(
    feasibility_estimator,
    graph_estimator,
    target_estimator=target_estimator,
    target_estimator_mode='regression',
    n_negative_per_positive=5,
    n_replicates=10,
    beam_size=2,
    fit_n_jobs=-1,
    fit_backend='loky',
    verbose=True,
    seed=0,
).fit(fit_graphs)


In [ ]:
generator.fit_target_estimator(fit_graphs, fit_targets)

In [ ]:
%%time
idx = random.randrange(len(fit_graphs))
graph = fit_graphs[idx]
print('Selecting random graph...')
_ = display_graphs([graph], n_graphs_per_line=1)

start_graph, target_n_edges = remove_edges(graph, size=.8)
desired_target = fit_targets[idx]
target_lambda = 0.5
print('Removed edges to create start graph with target_n_edges =', target_n_edges)
_ = display_graphs([start_graph], n_graphs_per_line=1)

print('Generating path to target_n_edges =', target_n_edges, 'and desired_target =', desired_target)
generation_path = generator.generate(
    start_graph,
    n_edges=target_n_edges,
    target=desired_target,
    target_lambda=target_lambda,
    draw_graphs_fn=lambda graphs, **kwargs: display_graphs(graphs, **kwargs),
)

print('n_fit_graphs =', len(fit_graphs))
print('start_n_edges =', start_graph.number_of_edges())
print('target_n_edges =', target_n_edges)
print('desired_target =', desired_target)
print('generation_path_length =', len(generation_path))
_ = display_graphs(generation_path, n_graphs_per_line=min(len(generation_path), 7))
_ = display_graphs([graph], n_graphs_per_line=1)


In [ ]:
size = 0.7

from abstractgraph_generative.edge_generator import mix_connected_components

idx_a, idx_b = random.sample(range(len(fit_graphs)), k=2)
graph_a, graph_b = fit_graphs[idx_a], fit_graphs[idx_b]
target_a, target_b = fit_targets[idx_a], fit_targets[idx_b]

_ = display_graphs([graph_a, graph_b], n_graphs_per_line=2)

start_graph_a, target_n_edges_a = remove_edges(graph_a, size=size)
start_graph_b, target_n_edges_b = remove_edges(graph_b, size=size)

mixed_graph = mix_connected_components(start_graph_a, start_graph_b, seed=0)
mixed_target_n_edges = int(np.mean([target_n_edges_a, target_n_edges_b]))
mixed_target = int(np.mean([target_a, target_b]))

print(f"mixed_target_n_edges = {mixed_target_n_edges}")
print(f"mixed_target = {mixed_target}")

_ = display_graphs([start_graph_a, start_graph_b, mixed_graph], n_graphs_per_line=3)


In [ ]:
generation_path = generator.generate(
    mixed_graph,
    n_edges=mixed_target_n_edges,
    target=mixed_target,
    target_lambda=0.5,
    draw_graphs_fn=lambda graphs, **kwargs: display_graphs(graphs, **kwargs),
)
print('n_fit_graphs =', len(fit_graphs))
print('start_n_edges =', start_graph.number_of_edges())
print('target_n_edges =', target_n_edges)
print('mixed_target =', mixed_target)
print('generation_path_length =', len(generation_path))
_ = display_graphs(generation_path, n_graphs_per_line=min(len(generation_path), 7))
_ = display_graphs([graph_a, graph_b, mixed_graph], n_graphs_per_line=2)

---